In [ ]:
import pandas as pd
import pycountry

In [ ]:
from _helpers import mock_snakemake

snakemake = mock_snakemake("prepare_iron_ore")

### File paths

In [ ]:
# Input from retrieve_iron_ore rule
production_fn = snakemake.input.iron_ore
bus_locations_fn = snakemake.input.bus_locations

# Output
production_clustered_fn = snakemake.output.iron_ore

In [ ]:
# Load config
config = snakemake.config

### Get iron ore data

In [ ]:
# Read the CSV data
production = pd.read_csv(production_fn, index_col="ISO_A2")

In [ ]:
production

### Cluster per region

### Cluster countries per region
Map each country in the production data to its region using the region definitions from the config file.

In [ ]:
# Load region definitions from config (regions now uses ISO-3 codes directly)
regions = config['regions']

# Build ISO-2 → region mapping (production data uses ISO_A2 / 2-letter codes)
# Convert ISO-3 codes from config to ISO-2 via pycountry
iso_to_region = {}
for region, iso3_codes in regions.items():
    for iso3 in iso3_codes:
        country = pycountry.countries.get(alpha_3=iso3)
        if country:
            iso_to_region[country.alpha_2] = region

# Map each row in production to its region
production['region'] = production.index.map(lambda iso: iso_to_region.get(iso, 'Other'))

### Aggregate iron ore production by region

In [ ]:
# Group by region: sum production and calculate weighted average price
# We need to do this in two steps since groupby.agg with custom function on single column doesn't have access to other columns

# First, sum the production
production_sum = production.groupby("region")["IronOreProductionMt"].sum()

# Then calculate weighted average cost (using the new column name)
production_avg_cost = production.groupby("region").apply(
    lambda group: (
        (group["IronOreEur/t_ironore"].fillna(0) * group["IronOreProductionMt"]).sum()
        / group["IronOreProductionMt"].sum()
        if group["IronOreProductionMt"].sum() > 0
        else 0
    )
)

# Combine into a single DataFrame
production_by_region = pd.DataFrame(
    {"IronOreProductionMt": production_sum, "IronOreEur/t_ironore": production_avg_cost}
)

# Ensure all regions from config are present, fill missing with 0
all_regions = pd.Series(0, index=regions.keys())
production_by_region = production_by_region.reindex(all_regions.index, fill_value=0)

# Reset index to make region a column
production_by_region = production_by_region.reset_index()
production_by_region = production_by_region.rename(columns={"index": "region"})

production_by_region

### Save clustered iron ore production data

In [ ]:
production_by_region = production_by_region.set_index("region")
production_by_region["IronOreEur/t_ironore"] = production_by_region[
    "IronOreEur/t_ironore"
].round(2)
production_by_region.to_csv(production_clustered_fn)